# Multi-Modal Models

## Introduction

So far in this course, we have built neural networks that process a single type of data: feedforward networks for tabular data, convolutional networks for images, and Transformers for text. But much of human reasoning is inherently multi-modal: a doctor reads a radiology report (text) while looking at the X-ray (image); a pathologist examines a tissue slide (image) alongside genomic data (tabular). **Multi-modal models** are neural networks that can process and relate multiple types of data, most commonly images and text.

In this lecture, we will cover:

* How images become tokens: the Vision Transformer (ViT)
* Connecting images and text: CLIP and contrastive learning
* Vision-Language Models: how modern systems (GPT-4o, Claude, Gemini) process images alongside text
* Hands-on: using multi-modal APIs for image understanding
* Biomedical applications: medical imaging, pathology, and clinical text
* Beyond vision and language: audio, video, and other modalities

You already know the Transformer architecture, self-attention, tokenization, and the LLM training pipeline from the previous lectures. The key new idea in this lecture is that we can extend the same Transformer architecture to process images (and other modalities) by converting them into sequences of tokens, just like text.

![xkcd 1425: Tasks](https://imgs.xkcd.com/comics/tasks.png)

*In 2014, this xkcd comic joked that identifying whether a photo contains a bird would require "a research team and five years." By 2024, AI-powered bird-identifying binoculars were commercially available. Multi-modal models are a big part of how we got here.*

## Vision Transformers (ViT)

### From CNNs to Transformers for images

In the deep learning lecture, we saw that **convolutional neural networks (CNNs)** process images using local filters that slide across the image. CNNs have built-in inductive biases for images: locality (nearby pixels matter more) and translation invariance (a cat is a cat regardless of where it appears).

The **Vision Transformer (ViT)** takes a radically different approach: treat an image as a sequence of tokens and apply a standard Transformer encoder. ViT was introduced in 2020 by Dosovitskiy et al. in the paper "An Image is Worth 16x16 Words."

The key insight is that just as text is a sequence of word/subword tokens, an image can be treated as a sequence of **patch tokens**.

### Patch embedding: images as sequences

ViT processes an image in four steps:

1. **Split** the image into fixed-size patches (e.g., 16x16 pixels)
2. **Flatten** each patch into a vector
3. **Project** each flattened patch through a linear layer to get a patch embedding
4. **Add** positional embeddings so the model knows where each patch came from

![ViT architecture (Dive into Deep Learning)](https://d2l.ai/_images/vit.svg)

*The Vision Transformer architecture. An image is split into patches, each patch is linearly embedded, positional encodings are added, and the resulting sequence is fed into a standard Transformer encoder. A special [CLS] token is prepended and its output is used for classification.*

For a 224x224 image with 16x16 patches, we get $224/16 = 14$ patches per side, so $14 \times 14 = 196$ patch tokens. Each patch is flattened from a $16 \times 16 \times 3 = 768$-dimensional vector (for RGB images) and linearly projected to the model's hidden dimension.

This is directly analogous to tokenization in NLP:

| NLP | Vision (ViT) |
|-----|-------------|
| Word/subword | Image patch (16x16 pixels) |
| Token embedding | Patch embedding (linear projection) |
| Positional encoding | Positional encoding |
| Sequence of tokens | Sequence of patches |
| [CLS] token for classification | [CLS] token for classification |

Let's implement patch embedding from scratch:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

Path("figs").mkdir(exist_ok=True)

# Create a synthetic 224x224 RGB image with colored shapes
np.random.seed(42)
img_array = np.ones((224, 224, 3)) * 0.9  # light gray background

# Blue circle
y, x = np.ogrid[:224, :224]
mask = (x - 80)**2 + (y - 80)**2 < 40**2
img_array[mask] = [0.2, 0.4, 0.8]

# Green rectangle
img_array[120:180, 100:180] = [0.2, 0.7, 0.3]

# Red triangle
for i in range(60):
    img_array[160-i, 30+i//2:90-i//2] = [0.8, 0.2, 0.2]

print(f"Image shape: {img_array.shape}")  # (224, 224, 3)

In [ ]:
# Split the image into 16x16 patches
patch_size = 16
h, w, c = img_array.shape
n_patches_h = h // patch_size  # 14
n_patches_w = w // patch_size  # 14
n_patches = n_patches_h * n_patches_w  # 196

# Extract patches
patches = img_array.reshape(
    n_patches_h, patch_size, n_patches_w, patch_size, c
)
patches = patches.transpose(0, 2, 1, 3, 4)  # (14, 14, 16, 16, 3)
patches = patches.reshape(n_patches, patch_size * patch_size * c)  # (196, 768)

print(f"Number of patches: {n_patches}")
print(f"Each patch flattened to: {patches.shape[1]} dimensions")
print(f"  = {patch_size} x {patch_size} x {c} (height x width x channels)")

In [ ]:
# Visualize the patches
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original image
axes[0].imshow(img_array)
axes[0].set_title("Original image (224 x 224)")
axes[0].axis("off")

# Show patch grid overlay
axes[1].imshow(img_array)
for i in range(1, n_patches_h):
    axes[1].axhline(y=i * patch_size, color="red", linewidth=0.5)
for j in range(1, n_patches_w):
    axes[1].axvline(x=j * patch_size, color="red", linewidth=0.5)
axes[1].set_title(f"Split into {n_patches_h} x {n_patches_w} = {n_patches} patches")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("figs/multi-modal-patches.png", dpi=150, bbox_inches="tight")

In [ ]:
# Visualize individual patches
fig, axes = plt.subplots(3, 5, figsize=(10, 6))
sample_indices = [0, 3, 7, 10, 13, 28, 42, 98, 105, 120, 140, 168, 175, 182, 195]

for ax, idx in zip(axes.flat, sample_indices):
    patch_img = patches[idx].reshape(patch_size, patch_size, c)
    ax.imshow(patch_img)
    row, col = idx // n_patches_w, idx % n_patches_w
    ax.set_title(f"Patch {idx}\n({row},{col})", fontsize=8)
    ax.axis("off")

plt.suptitle("Individual 16x16 patches", fontsize=12)
plt.tight_layout()
plt.savefig("figs/multi-modal-individual-patches.png", dpi=150, bbox_inches="tight")

In [ ]:
# Linear projection: project each 768-dim patch to a d_model-dim embedding
d_model = 64  # small for demonstration

np.random.seed(42)
W_proj = np.random.randn(patch_size * patch_size * c, d_model) / np.sqrt(patch_size * patch_size * c)  # (768, 64)
b_proj = np.zeros(d_model)

patch_embeddings = patches @ W_proj + b_proj  # (196, 64)
print(f"Patch embeddings shape: {patch_embeddings.shape}")

# Add a [CLS] token (learnable embedding)
cls_token = np.random.randn(1, d_model) / np.sqrt(d_model)  # (1, 64)
sequence = np.concatenate([cls_token, patch_embeddings], axis=0)  # (197, 64)
print(f"Full sequence (with [CLS]): {sequence.shape}")
print(f"  = 1 [CLS] token + {n_patches} patch tokens")

Notice that the entire process is just reshaping and a matrix multiplication. The splitting and linear projection can be implemented as a single convolution with kernel size = stride = patch size.

### Why does ViT work?

ViT works because self-attention lets every patch attend to every other patch. A CNN with a 3x3 filter can only see a 3x3 neighborhood at each layer. To relate distant parts of an image, a CNN needs many stacked layers to gradually expand its receptive field. Self-attention relates all patches to all other patches in a single layer.

![CNN vs ViT: local filters vs global attention](figs/multi-modal-cnn-vs-vit.png)

*Left: a CNN filter (blue) only sees a small local neighborhood around each pixel. Distant pixels (gray) are unreachable in a single layer. Right: in ViT, each patch (red) attends to every other patch in a single self-attention layer, giving it a global receptive field from the start.*

However, ViT lacks the inductive biases of CNNs (locality, translation invariance), so it needs much more training data to learn these properties from scratch. The original ViT was pretrained on 300 million images (JFT-300M). With enough data, ViT matches or exceeds CNNs on image classification benchmarks.

### Question

Consider the following code that creates a ViT-style patch embedding:

In [ ]:
img = np.random.randn(224, 224, 3)
patch_size = 16
n_patches = (224 // patch_size) ** 2

1. What is the value of `n_patches`?
2. If we change `patch_size = 32`, what is `n_patches` now? What is the tradeoff?
3. In the LLM lecture, we discussed that self-attention has $O(n^2)$ complexity in the sequence length. If we process a 1024x1024 image with `patch_size = 16`, how many tokens do we get, and how does the attention cost compare to the 224x224 case?

### Answer

1. $(224/16)^2 = 14^2 = 196$ patch tokens.
2. $(224/32)^2 = 7^2 = 49$ tokens. Fewer tokens means faster attention computation ($O(49^2)$ vs $O(196^2)$, a 16x reduction), but each patch is larger (32x32 = 1024 pixels), so the model sees coarser features. This tradeoff between resolution and computational cost is fundamental in ViT design.
3. A 1024x1024 image with 16x16 patches produces $(1024/16)^2 = 4096$ tokens. Self-attention on 4096 tokens costs $O(4096^2) \approx 16.8M$ operations per attention layer, compared to $O(196^2) \approx 38K$ for 224x224. That is a 440x increase. This is why high-resolution image processing often uses hierarchical approaches (process patches at multiple scales) or windowed attention.

## Connecting Vision and Language: CLIP

### The alignment problem

We now have two types of Transformers: one for text (GPT, BERT) and one for images (ViT). But they live in separate embedding spaces. The word "cat" and a photo of a cat are represented by completely different vectors with no relationship to each other.

**CLIP** (Contrastive Language-Image Pre-training), introduced by OpenAI in 2021, solves this by training an image encoder and a text encoder *jointly* so that matching image-text pairs have similar embeddings.

### Contrastive learning

CLIP's training objective is elegant. Given a batch of $N$ image-text pairs:

1. Encode all $N$ images through the image encoder (ViT) to get image embeddings $\{v_1, \ldots, v_N\}$
2. Encode all $N$ texts through the text encoder (Transformer) to get text embeddings $\{t_1, \ldots, t_N\}$
3. Compute the $N \times N$ matrix of cosine similarities between all image-text pairs
4. The training objective pushes the $N$ matching pairs (diagonal) to have high similarity and the $N^2 - N$ non-matching pairs (off-diagonal) to have low similarity

![CLIP architecture (OpenAI)](https://github.com/openai/CLIP/raw/main/CLIP.png)

*The CLIP training process. An image encoder and text encoder are trained jointly on image-text pairs using a contrastive objective. The diagonal of the similarity matrix (matching pairs) is maximized while off-diagonal entries (non-matching pairs) are minimized.*

The loss function is a symmetric cross-entropy over the similarity matrix. Let $s_{ij} = \cos(v_i, t_j) / \tau$ be the cosine similarity between image embedding $v_i$ and text embedding $t_j$, scaled by a learned temperature $\tau$. The image-to-text loss treats each row of the similarity matrix as an $N$-way classification problem (for image $i$, which of the $N$ texts is the correct match?):

$$\mathcal{L}_{\text{image}\to\text{text}} = -\frac{1}{N}\sum_{i=1}^{N} \log \frac{\exp(s_{ii})}{\sum_{j=1}^{N} \exp(s_{ij})}$$

The text-to-image loss does the same over columns (for text $j$, which image matches?):

$$\mathcal{L}_{\text{text}\to\text{image}} = -\frac{1}{N}\sum_{j=1}^{N} \log \frac{\exp(s_{jj})}{\sum_{i=1}^{N} \exp(s_{ij})}$$

The total CLIP loss is the average of both directions:

$$\mathcal{L} = \frac{1}{2}\left(\mathcal{L}_{\text{image}\to\text{text}} + \mathcal{L}_{\text{text}\to\text{image}}\right)$$

Each term is just softmax cross-entropy with the diagonal entry as the correct class. Maximizing the log probability of the diagonal entry simultaneously pushes matching pairs together and non-matching pairs apart.

Let's implement this:

In [ ]:
np.random.seed(42)

# Simulate CLIP embeddings for a batch of 5 image-text pairs
N = 5
d_embed = 128

# In practice, these come from the image and text encoders
image_embeddings = np.random.randn(N, d_embed)
text_embeddings = np.random.randn(N, d_embed)

# Normalize to unit vectors (CLIP uses cosine similarity)
image_embeddings = image_embeddings / np.linalg.norm(image_embeddings, axis=1, keepdims=True)
text_embeddings = text_embeddings / np.linalg.norm(text_embeddings, axis=1, keepdims=True)

# Compute cosine similarity matrix
# (in CLIP, this is scaled by a learned temperature parameter)
temperature = 0.07
similarity = (image_embeddings @ text_embeddings.T) / temperature

print("Cosine similarity matrix (scaled by 1/temperature):")
print(np.round(similarity, 2))
print(f"\nShape: {similarity.shape} (N images x N texts)")
print("Diagonal = matching pairs, off-diagonal = non-matching pairs")

In [ ]:
# Visualize the similarity matrix before and after training
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Before training: random similarities
labels = ["A chest X-ray\nshowing pneumonia",
          "A photo of\na golden retriever",
          "A histology\nslide of tissue",
          "A brain MRI\nscan",
          "A photo of\na sunset"]
short_labels = ["X-ray", "Dog", "Histology", "MRI", "Sunset"]

# Random (untrained) similarity
random_sim = np.random.randn(N, N) * 0.3
axes[0].imshow(random_sim, cmap="RdBu_r", vmin=-3, vmax=3)
axes[0].set_xticks(range(N))
axes[0].set_yticks(range(N))
axes[0].set_xticklabels(short_labels, fontsize=8)
axes[0].set_yticklabels(short_labels, fontsize=8)
axes[0].set_xlabel("Text")
axes[0].set_ylabel("Image")
axes[0].set_title("Before training\n(random similarities)")
for i in range(N):
    for j in range(N):
        axes[0].text(j, i, f"{random_sim[i,j]:.1f}", ha="center", va="center", fontsize=8)

# After training: diagonal should be high, off-diagonal low
trained_sim = np.eye(N) * 2.5 + np.random.randn(N, N) * 0.3
axes[1].imshow(trained_sim, cmap="RdBu_r", vmin=-3, vmax=3)
axes[1].set_xticks(range(N))
axes[1].set_yticks(range(N))
axes[1].set_xticklabels(short_labels, fontsize=8)
axes[1].set_yticklabels(short_labels, fontsize=8)
axes[1].set_xlabel("Text")
axes[1].set_ylabel("Image")
axes[1].set_title("After CLIP training\n(matching pairs aligned)")
for i in range(N):
    for j in range(N):
        axes[1].text(j, i, f"{trained_sim[i,j]:.1f}", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.savefig("figs/multi-modal-clip-similarity.png", dpi=150, bbox_inches="tight")

![Chihuahua or Muffin? (Karen Zack / @teenybiscuit)](https://static.boredpanda.com/blog/wp-content/uploads/2016/03/dog-food-comparison-bagel-muffin-lookalike-teenybiscuit-karen-zack-7__700.jpg)

![Labradoodle or Fried Chicken? (Karen Zack / @teenybiscuit)](https://static.boredpanda.com/blog/wp-content/uploads/2016/03/dog-food-comparison-bagel-muffin-lookalike-teenybiscuit-karen-zack-5__700.jpg)

*Chihuahuas vs muffins. Labradoodles vs fried chicken.*

### Zero-shot classification with CLIP

Once trained, CLIP enables **zero-shot image classification** without any task-specific training. To classify an image into one of $K$ categories:

1. Encode the image to get an image embedding
2. Create text prompts for each category: "a photo of a {category}"
3. Encode all text prompts to get text embeddings
4. Choose the category whose text embedding has the highest cosine similarity with the image embedding

This is powerful because you can change the categories at inference time without retraining.

In [ ]:
# pip install transformers torch pillow
import torch
from PIL import Image
import requests
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel

# Load CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Download a sample image (cat photo from Wikipedia)
url = "https://upload.wikimedia.org/wikipedia/commons/4/4d/Cat_November_2010-1a.jpg"
response = requests.get(url, headers={"User-Agent": "BIOS735/1.0"})
image = Image.open(BytesIO(response.content)).resize((224, 224))

# Define candidate labels
labels = ["a photo of a cat", "a photo of a dog", "a chest X-ray",
          "a histology slide", "a photo of a car"]

# Compute similarities
inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    outputs = clip_model(**inputs)

# Get probabilities
logits = outputs.logits_per_image  # (1, num_labels)
probs = logits.softmax(dim=1).numpy()[0]

for label, prob in sorted(zip(labels, probs), key=lambda x: -x[1]):
    print(f"  {label:30s}: {prob:.4f}")

In [ ]:
# Visualize zero-shot classification
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(image)
axes[0].axis("off")
axes[0].set_title("Input image")

short_labels = [l.replace("a photo of ", "").replace("a ", "") for l in labels]
colors = ["#2ecc71" if p == max(probs) else "#95a5a6" for p in probs]
sorted_idx = np.argsort(probs)
axes[1].barh(range(len(labels)), probs[sorted_idx], color=[colors[i] for i in sorted_idx])
axes[1].set_yticks(range(len(labels)))
axes[1].set_yticklabels([short_labels[i] for i in sorted_idx])
axes[1].set_xlabel("Probability")
axes[1].set_title("CLIP zero-shot classification")

plt.tight_layout()
plt.savefig("figs/multi-modal-clip-zeroshot.png", dpi=150, bbox_inches="tight")

CLIP was trained on 400 million image-text pairs from the internet. It never saw explicit category labels during training, yet it can classify images into arbitrary categories by leveraging the learned alignment between vision and language.

Because CLIP learns a semantic embedding space (not just word matching), it understands synonyms and even cross-lingual concepts. Let's test this:

In [ ]:
import matplotlib
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Arial Unicode MS", "DejaVu Sans"]
neg = {"a photo of a dog", "a chest X-ray", "a histology slide", "a photo of a car"}

def plot_clip_probs(labels, probs, title, color="#2ecc71"):
    """Helper to plot CLIP zero-shot probabilities."""
    short = [l.replace("a photo of a ", "").replace("a photo of an ", "") for l in labels]
    colors = ["#95a5a6" if l in neg else color for l in labels]
    sorted_idx = np.argsort(probs)
    fig, ax = plt.subplots(figsize=(7, max(3, len(labels) * 0.45)))
    ax.barh(range(len(labels)), probs[sorted_idx],
            color=[colors[i] for i in sorted_idx])
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels([short[i] for i in sorted_idx])
    ax.set_xlabel("Probability")
    ax.set_title(title)
    plt.tight_layout()

In [ ]:
# Synonyms: does CLIP know that "feline" and "kitten" are related to "cat"?
synonym_labels = ["a photo of a cat", "a photo of a feline", "a photo of a kitten",
                  "some other word for cat that i can't think of",
                  "a photo of a dog", "a chest X-ray",
                  "a histology slide", "a photo of a car"]

inputs = clip_processor(text=synonym_labels, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    outputs = clip_model(**inputs)
probs_syn = outputs.logits_per_image.softmax(dim=1).numpy()[0]

print("Synonyms test:")
for label, prob in sorted(zip(synonym_labels, probs_syn), key=lambda x: -x[1]):
    print(f"  {label:50s}: {prob:.4f}")

plot_clip_probs(synonym_labels, probs_syn, "English synonyms")

All three cat-related labels ("cat", "feline", "kitten") score well above the negative categories, confirming that the embedding space captures semantic meaning, not just surface-level word matching. Even the vague phrase "some other word for cat that I can't think of" scores above "dog", because CLIP's text encoder understands that this sentence is *about* cats. The model is not doing keyword lookup; it is computing a semantic representation of the entire sentence.

In [ ]:
# Cross-lingual: does CLIP understand "cat" in other languages?
cross_lingual_labels = ["a photo of a cat", "a photo of a gato",
                        "a photo of a 猫",
                        "a photo of a dog", "a chest X-ray",
                        "a histology slide", "a photo of a car"]

inputs = clip_processor(text=cross_lingual_labels, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    outputs = clip_model(**inputs)
probs_lang = outputs.logits_per_image.softmax(dim=1).numpy()[0]

print("Cross-lingual test:")
for label, prob in sorted(zip(cross_lingual_labels, probs_lang), key=lambda x: -x[1]):
    print(f"  {label:50s}: {prob:.4f}")

plot_clip_probs(cross_lingual_labels, probs_lang, "Cross-lingual (cat in other languages)")

"Gato" (Spanish) and "猫" (Chinese) both score comparably to "cat", with all three far above the negatives. CLIP was trained on predominantly English data, but its training corpus (400 million image-text pairs from the internet) included enough multilingual content for the model to learn cross-lingual associations. This is an emergent capability: CLIP was not explicitly trained to be multilingual, but the shared embedding space naturally aligns concepts across languages when the training data contains them.

In [ ]:
# Semantic hierarchy: does CLIP distinguish specificity levels?
hierarchy_labels = ["a photo of a cat", "a photo of a mammal", "a photo of an animal",
                    "a photo of a living thing",
                    "a photo of a dog", "a chest X-ray",
                    "a histology slide", "a photo of a car"]

inputs = clip_processor(text=hierarchy_labels, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    outputs = clip_model(**inputs)
probs_hier = outputs.logits_per_image.softmax(dim=1).numpy()[0]

print("Semantic hierarchy test:")
for label, prob in sorted(zip(hierarchy_labels, probs_hier), key=lambda x: -x[1]):
    print(f"  {label:50s}: {prob:.4f}")

plot_clip_probs(hierarchy_labels, probs_hier, "Semantic hierarchy", color="#e67e22")

The semantic hierarchy test reveals how CLIP handles different levels of specificity. "Cat" (the most specific and correct label) scores highest, followed by "animal", "mammal", and "living thing" in decreasing order, since each is a correct but increasingly vague description of the image.

### Question

Consider the CLIP similarity computation from the code above:

In [ ]:
N = 32768
similarity = image_embeddings @ text_embeddings.T  # shape: (N, N)
# Training: maximize similarity[i, i], minimize similarity[i, j] for j != i

1. For each image, how many negative pairs (non-matching texts) does it have? How does this compare to the number of positive pairs?
2. What would happen if we trained with `N = 2` instead? Why does batch size matter for contrastive learning?
3. CLIP can classify images into categories it was never explicitly trained on (zero-shot). How is this possible, given that the model was only trained to match images with their captions?

### Answer

1. Each image has 1 positive pair (its matching text) and $N - 1 = 32{,}767$ negative pairs. The ratio of negatives to positives is 32,767:1.
2. With $N = 2$, each image only has 1 negative pair to contrast against. The model can easily distinguish 2 options without learning meaningful representations. With large $N$, the model must learn fine-grained features to distinguish the correct text from thousands of alternatives. Larger batches provide harder negatives (more texts that might be somewhat related but are not the exact match), forcing the model to learn more discriminative embeddings.
3. During training, CLIP learns a shared embedding space where semantically similar images and texts are close together. The text "a photo of a cat" is close to images of cats in this space, even if that exact phrase never appeared as a training caption. The model generalizes from captions like "my cute tabby cat sleeping on the couch" to the general concept of "cat." Zero-shot classification works because natural language is flexible enough to describe any visual category.

## Vision-Language Models (VLMs)

### From CLIP to conversational vision

CLIP aligns images and text in a shared embedding space, but it cannot have a conversation about an image. It can match and rank, but not generate. **Vision-Language Models (VLMs)** combine a vision encoder with an LLM to enable open-ended visual question answering, image description, and reasoning about images.

Modern VLMs like GPT-4o, Claude, and Gemini can:
- Describe what is in an image
- Answer questions about image content
- Read and extract text from documents and charts
- Reason about spatial relationships
- Analyze medical images (with appropriate caveats)

### Architecture: how VLMs process images

The typical VLM architecture has three components:

1. **Vision encoder**: Extracts visual features from the image (usually a pretrained ViT, often CLIP's image encoder)
2. **Projection layer**: Maps vision encoder outputs to the LLM's embedding space (an MLP or cross-attention module)
3. **LLM decoder**: Generates text responses conditioned on both the visual tokens and the text prompt

```mermaid
flowchart LR
    subgraph Vision
        A["Image<br/>(224x224)"] --> B["Vision Encoder<br/>(ViT)"]
        B --> C["Patch tokens<br/>(196 x d_vision)"]
    end
    subgraph Projection
        C --> D["MLP Projector"]
        D --> E["Visual tokens<br/>(196 x d_llm)"]
    end
    subgraph Language
        F["Text prompt<br/>tokens"] --> G["LLM Decoder<br/>(Transformer)"]
        E --> G
        G --> H["Generated<br/>response"]
    end
```

The key idea is that after the projection layer, visual tokens occupy the same vector space as text tokens. The LLM decoder receives a single concatenated sequence: visual tokens first, then text prompt tokens. For example, if the image produces 196 visual tokens and the prompt "What is this?" is 5 text tokens, the LLM sees a sequence of 201 tokens.

Generation then works exactly like in a text-only LLM (recall the autoregressive generation from the LLM lecture):

1. The full 201-token sequence passes through the Transformer's self-attention layers. Each token attends to all previous tokens, so text tokens can attend to visual tokens (this is how the model "sees" the image).
2. The LLM predicts the next token after the last prompt token, conditioned on both the visual and text context.
3. The predicted token is appended to the sequence (now 202 tokens), and the process repeats until the model generates a stop token.

The visual tokens act like a "prefix" that the LLM conditions on but never generates. They inject image information into the same attention mechanism the model already uses for text. No architectural change to the Transformer is needed; the only new components are the vision encoder and the projection layer.

**Why all patch tokens, not just the [CLS] token?** In the ViT section, we saw that the [CLS] token is used for image classification: the entire image is compressed into a single vector that is fed to a classifier. This works for classification ("is this a cat or a dog?") but discards spatial information. A VLM needs to answer questions like "what color is the object in the top-left corner?" or "read the text on the sign," which require fine-grained spatial detail. By passing all patch tokens to the LLM, each patch retains information about its local region of the image, and the LLM's attention mechanism can selectively focus on the relevant patches when generating each word of the response.

### LLaVA: a concrete example

**LLaVA** (Large Language and Vision Assistant) is an influential open-source VLM that clearly illustrates this architecture. It uses:

- **Vision encoder**: CLIP ViT-L/14 (produces 576 visual tokens per image)
- **Projector**: A 2-layer MLP that maps from the vision encoder's 1,024 dimensions to the LLM's 4,096 dimensions
- **LLM**: Vicuna (a fine-tuned LLaMA model)

Training happens in two stages:

1. **Stage 1 (Pretraining)**: Freeze the vision encoder and LLM, train only the projector on 600K image-caption pairs. This teaches the projector how to translate visual features into the LLM's "language."
2. **Stage 2 (Fine-tuning)**: Unfreeze the projector and LLM (keep vision encoder frozen), fine-tune on 150K visual instruction-following examples (e.g., "What is unusual about this image?" with a detailed answer).

```mermaid
flowchart LR
    A["Image<br/>224 x 224 x 3<br/>(150,528 values)"] -->|"split into patches<br/>+ linear projection"| B["ViT patch tokens<br/>576 x 1,024"]
    B -->|"MLP projector"| C["Visual tokens<br/>576 x 4,096"]
    D["Text prompt<br/>~100 x 4,096"] --> E["Combined sequence<br/>~676 x 4,096"]
    C --> E
    E -->|"LLM Decoder"| F["Generated<br/>response"]
```

### Using multi-modal APIs

Let's use a multi-modal model via the OpenRouter API (which provides an OpenAI-compatible interface to many models):

In [ ]:
# pip install openai
import os
import base64
import requests
from openai import OpenAI
from io import BytesIO

# OpenRouter uses the OpenAI-compatible API format
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
model_name = "nvidia/nemotron-nano-12b-v2-vl:free"

# Helper: download an image, resize, and encode as a data URL
def image_to_data_url(url, max_size=1024):
    response = requests.get(url, headers={"User-Agent": "BIOS735/1.0"})
    img = Image.open(BytesIO(response.content))
    if max(img.size) > max_size:
        img.thumbnail((max_size, max_size))
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=85)
    b64 = base64.standard_b64encode(buf.getvalue()).decode("utf-8")
    return img, f"data:image/jpeg;base64,{b64}"

# Download a sample image
image_url = "https://upload.wikimedia.org/wikipedia/commons/b/b6/Image_created_with_a_mobile_phone.png"
sample_img, sample_data_url = image_to_data_url(image_url)

# Display the image so we can see what the model sees
plt.figure(figsize=(5, 5))
plt.imshow(sample_img)
plt.axis("off")
plt.title("Input image")
plt.tight_layout()

In [ ]:
# Ask the model to describe and analyze the image
response = client.chat.completions.create(
    model=model_name,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": sample_data_url}},
            {"type": "text", "text": "Describe this image in 2-3 sentences. Then list the main objects you see."},
        ]
    }],
    max_tokens=300,
)
print(response.choices[0].message.content)

Notice the message format: the `content` field is a **list** containing both `image_url` and `text` blocks. The image is encoded as a base64 data URL (`data:image/jpeg;base64,...`). This is the OpenAI-compatible format used by many VLM providers. The model processes the image tokens and text tokens together through its Transformer layers.

In [ ]:
# Structured extraction from an image
response = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a visual analysis assistant. Always respond in valid JSON."},
        {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": sample_data_url}},
            {"type": "text", "text": "Analyze this image and return a JSON object with keys: 'description' (1 sentence), 'objects' (list of strings), 'dominant_colors' (list of strings), 'scene_type' (indoor/outdoor/other)."},
        ]}
    ],
    max_tokens=500,
)
print(response.choices[0].message.content)

In [ ]:
# Comparing two images
cat_img, cat_data_url = image_to_data_url("https://upload.wikimedia.org/wikipedia/commons/4/4d/Cat_November_2010-1a.jpg")
dog_img, dog_data_url = image_to_data_url("https://upload.wikimedia.org/wikipedia/commons/2/26/YellowLabradorLooking_new.jpg")

# Display both images side by side
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cat_img)
axes[0].axis("off")
axes[0].set_title("Image 1")
axes[1].imshow(dog_img)
axes[1].axis("off")
axes[1].set_title("Image 2")
plt.tight_layout()

In [ ]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": cat_data_url}},
            {"type": "image_url", "image_url": {"url": dog_data_url}},
            {"type": "text", "text": "Compare these two images. What are the similarities and differences?"},
        ]
    }],
    max_tokens=300,
)
print(response.choices[0].message.content)

### Question

Consider this VLM setup:

In [ ]:
image_size = 384
patch_size = 16
n_visual_tokens = (image_size // patch_size) ** 2
text_tokens = 50
total_tokens = n_visual_tokens + text_tokens

1. What is `n_visual_tokens`?
2. What is `total_tokens`? What fraction of the total sequence is visual?
3. A camera photo is 4000x3000 pixels. If we used `patch_size = 4` instead of resizing the image, how many visual tokens would that produce? Why is this impractical?

### Answer

1. $(384/16)^2 = 24^2 = 576$ visual tokens.
2. Total sequence = 576 visual + 50 text = 626 tokens. Visual fraction = 576/626 = 92%. The sequence is dominated by visual tokens, which is why image processing is computationally expensive in VLMs.
3. With 4x4 patches on a 4000x3000 image, we would get $(4000/4) \times (3000/4) = 1000 \times 750 = 750{,}000$ tokens. Self-attention has $O(n^2)$ complexity, so processing 750K tokens would require $\sim 5.6 \times 10^{11}$ operations per attention layer, which is computationally prohibitive. Resizing the image or using larger patches is a necessary tradeoff between resolution and computational feasibility.

## Biomedical Applications

Multi-modal models are particularly impactful in biomedicine, where clinicians routinely integrate multiple data types (images, text, genomics) for diagnosis and treatment.

### Medical image understanding

Traditional medical image analysis uses task-specific CNNs: one model for chest X-ray classification, another for retinal disease detection, another for skin lesion classification. Each model must be trained on a large labeled dataset for that specific task.

Multi-modal models change this paradigm. **BiomedCLIP** (Zhang et al., 2023), trained on 15 million image-text pairs from PubMed Central, can:

- Classify medical images across modalities (X-rays, CT scans, histology slides, dermatology photos) via zero-shot classification
- Retrieve relevant images given text queries ("show me chest X-rays with pleural effusion")

Note that BiomedCLIP is a contrastive model (like CLIP), so it excels at retrieval and classification. For open-ended visual question answering, you need a VLM with a language decoder (like LLaVA-Med or GPT-4o).

In [ ]:
# Example: Using a VLM for medical image analysis
# (In practice, always validate with clinical expertise)

medical_prompt = """You are a medical imaging teaching assistant.
Describe what you observe in this image as if teaching a student.
If this is a medical image, use proper terminology.
If it is not a medical image, explain what a medical version would look like.
Note: This is for educational purposes only, not clinical diagnosis."""

# Use the same sample image for demonstration
# (In practice, you would use an actual medical image)
plt.figure(figsize=(5, 5))
plt.imshow(sample_img)
plt.axis("off")
plt.title("Input image for medical analysis")
plt.tight_layout()

In [ ]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": sample_data_url}},
            {"type": "text", "text": medical_prompt},
        ]
    }],
    max_tokens=600,
)
print(response.choices[0].message.content)

### Multi-modal data fusion in clinical research

Beyond single images, clinical research increasingly combines multiple data modalities:

| Modality combination | Application | Example |
|---------------------|-------------|---------|
| Radiology + clinical notes | Automated report generation | Generate structured reports from chest X-rays |
| Pathology + genomics | Cancer subtyping | Integrate H&E slides with mutation profiles |
| CT + lab values + notes | Treatment response prediction | Predict immunotherapy response in oncology |
| Retinal images + EHR | Systemic disease screening | Detect diabetes/cardiovascular risk from eye scans |

A consistent finding across recent studies is that multi-modal models combining imaging with other data types outperform single-modality models, often by a substantial margin. The intuition is straightforward: different modalities provide complementary information that no single source captures alone.

In [ ]:
# Simulate multi-modal fusion: combining image features with tabular data
np.random.seed(42)

n_patients = 200

# Simulated image features (from a pretrained vision encoder)
image_features = np.random.randn(n_patients, 64)

# Simulated clinical features
age = np.random.normal(65, 10, n_patients)
tumor_size = np.random.exponential(3, n_patients)
mutation_count = np.random.poisson(50, n_patients)
clinical_features = np.column_stack([age, tumor_size, mutation_count])

# Simple fusion: concatenate features
fused_features = np.concatenate([image_features, clinical_features], axis=1)
print(f"Image features:    {image_features.shape}")
print(f"Clinical features: {clinical_features.shape}")
print(f"Fused features:    {fused_features.shape}")

# Simulate treatment response (binary outcome)
# In reality, this would come from follow-up data
true_weights = np.random.randn(fused_features.shape[1]) * 0.1
logits = fused_features @ true_weights
probs = 1 / (1 + np.exp(-logits))
response = (probs > 0.5).astype(int)

print(f"\nResponse rate: {response.mean():.1%}")

In [ ]:
# Compare single-modality vs multi-modal prediction
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

models = {
    "Image only": image_features,
    "Clinical only": clinical_features,
    "Multi-modal (fused)": fused_features,
}

print("5-fold cross-validated AUC:")
print("-" * 40)
aucs = {}
for name, X in models.items():
    clf = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(clf, X, response, cv=5, scoring="roc_auc")
    aucs[name] = scores.mean()
    print(f"  {name:25s}: {scores.mean():.3f} +/- {scores.std():.3f}")

In [ ]:
# Visualize the comparison
fig, ax = plt.subplots(figsize=(7, 4))
names = list(aucs.keys())
values = list(aucs.values())
colors = ["#3498db", "#2ecc71", "#e74c3c"]
bars = ax.bar(names, values, color=colors, edgecolor="white", linewidth=2)
ax.set_ylabel("AUC (5-fold CV)")
ax.set_title("Single-modality vs. multi-modal prediction")
ax.set_ylim(0.4, 1.0)
ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=1, label="Random")
ax.legend()
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("figs/multi-modal-fusion-comparison.png", dpi=150, bbox_inches="tight")

### Question

In the fusion example above, we used early fusion (concatenation):

In [ ]:
fused_features = np.concatenate([image_features, clinical_features], axis=1)
# image_features: (200, 64), clinical_features: (200, 3)
# fused_features: (200, 67)

A research team wants to build a multi-modal model that predicts cancer prognosis from pathology slides (images) and genomic mutation profiles (tabular data).

1. The code above uses simple concatenation. Describe an alternative fusion approach and its tradeoffs.
2. The team has 500 patients with both pathology slides and genomic data, and 10,000 patients with only pathology slides. How might they leverage all available data?
3. Why is multi-modal fusion especially valuable in medicine, compared to domains like social media image captioning?

### Answer

1. The code uses **early fusion** (concatenation), which is simple but doesn't model interactions between modalities. An alternative is **cross-attention fusion**: use a Transformer where image tokens attend to genomic tokens (and vice versa), allowing the model to learn which visual features are relevant given the genetic context. This is more expressive but requires more data and compute to train.
2. **Pretraining on the larger dataset**: Pretrain the image encoder on the 10,000 pathology slides (e.g., using self-supervised learning or with any available labels). Then fine-tune the full multi-modal model on the 500 paired samples. This leverages the larger dataset to learn strong visual features before introducing the genomic modality.
3. In medicine, different modalities often provide **complementary** rather than redundant information. A pathology slide shows tissue morphology; genomic data reveals molecular drivers. Neither alone captures the full picture. In contrast, an image and its caption on social media are largely redundant (the caption describes what is in the image). The complementary nature of medical data means multi-modal fusion can provide genuinely new predictive power, not just confirmation of what a single modality already captures.

## Beyond Vision and Language

### Other modalities

While vision-language models are the most common, the multi-modal paradigm extends to many other data types:

| Modality | Tokenization approach | Example models |
|----------|-----------------------|---------------|
| **Audio/speech** | Mel spectrogram patches or learned audio tokens | Whisper, AudioPaLM |
| **Video** | Spatial-temporal patches (3D patches across frames) | VideoMAE, Gemini |
| **3D point clouds** | Point patches or voxel patches | Point-BERT |
| **Genomic sequences** | k-mer tokenization (similar to BPE for DNA) | DNABERT, Nucleotide Transformer |
| **Time series** | Fixed-length windows as patches | PatchTST |

The unifying idea is the same: convert each modality into a sequence of tokens and process them with a Transformer.

### The Transformer as a universal architecture

A remarkable trend in deep learning is that the Transformer architecture works well across nearly all modalities. The same core operation, self-attention, captures relationships between text tokens, image patches, audio frames, and video clips.

```mermaid
flowchart TB
    subgraph Modalities
        A["Text"] --> T["Tokenizer<br/>(BPE)"]
        B["Image"] --> U["Patchify<br/>(ViT)"]
        C["Audio"] --> V["Spectrogram<br/>patches"]
        D["Genomics"] --> W["k-mer<br/>tokenizer"]
    end
    subgraph "Shared Architecture"
        T --> X["Token<br/>sequence"]
        U --> X
        V --> X
        W --> X
        X --> Y["Transformer<br/>Encoder/Decoder"]
        Y --> Z["Output"]
    end
```

This universality is why models like Gemini and GPT-4o can process text, images, audio, and video in a single model: all modalities are converted to tokens and processed by the same Transformer. The modality-specific part is only the tokenizer/encoder at the input.

### From multi-modal understanding to generation

The models we have discussed so far primarily *understand* multiple modalities: they take in images and text, and produce text. A newer frontier is **multi-modal generation**: models that can produce images, audio, or video.

| Capability | Input | Output | Example |
|-----------|-------|--------|---------|
| Image captioning | Image | Text | CLIP + LLM, LLaVA |
| Visual QA | Image + Question | Text | GPT-4o, Claude |
| Text-to-image | Text | Image | DALL-E 3, Stable Diffusion |
| Text-to-speech | Text | Audio | ElevenLabs, Bark |
| Text-to-video | Text | Video | Sora, Veo |
| Any-to-any | Any modality | Any modality | Gemini, GPT-4o |

Text-to-image models (like DALL-E and Stable Diffusion) typically use **diffusion models** rather than autoregressive generation. We will cover diffusion models in the next lecture.

## Practical Considerations

### Computational costs

Multi-modal models are significantly more expensive than text-only models because images add many tokens to the sequence:

In [ ]:
# Compare token counts for different inputs
scenarios = {
    "Short text prompt (50 words)": 65,
    "Long text prompt (500 words)": 650,
    "One image (384x384, 16x16 patches)": 576,
    "One image + short text": 576 + 65,
    "Four images + text (document analysis)": 576 * 4 + 200,
}

fig, ax = plt.subplots(figsize=(9, 4))
names = list(scenarios.keys())
tokens = list(scenarios.values())
colors = ["#3498db" if "image" not in n.lower() else "#e74c3c" for n in names]
bars = ax.barh(range(len(names)), tokens, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Number of tokens")
ax.set_title("Token count comparison: text vs. multi-modal inputs")
for bar, tok in zip(bars, tokens):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            str(tok), va="center", fontsize=9)
plt.tight_layout()
plt.savefig("figs/multi-modal-token-comparison.png", dpi=150, bbox_inches="tight")

### Limitations and failure modes

Multi-modal models have specific failure modes beyond those of text-only LLMs:

1. **Hallucinating image content**: The model may describe objects that are not present in the image, especially small or ambiguous details.
2. **Poor spatial reasoning**: Models often struggle with precise counting ("how many people are in this photo?") and spatial relationships ("is the cup to the left or right of the plate?").
3. **OCR errors**: While models can read text in images, they may make errors on small, blurry, or stylized text.
4. **Limited resolution**: Images are typically downsampled before processing, so fine details may be lost.
5. **Bias and safety**: Models inherit biases from their training data, which may lead to unfair or stereotyped descriptions of people in images.
6. **Adversarial vulnerability**: Small, carefully crafted perturbations to an image can cause a model to misclassify it with high confidence, even though the change is invisible to humans.

![3D-printed adversarial turtle classified as a rifle (MIT CSAIL / labsix)](https://www.csail.mit.edu/sites/default/files/2017-11/RIFLE_TURTLE_1.JPG)

*MIT researchers 3D-printed a turtle with a subtly modified texture. Google's InceptionV3 classifier confidently identifies it as a "rifle" from every viewing angle. The perturbation is nearly invisible to humans but exploits learned pixel patterns in the network.*

In [ ]:
# Demonstrate a spatial reasoning task
spatial_prompt = """Look at this image and answer precisely:
1. How many distinct objects can you count?
2. Describe the spatial layout (what is above, below, left, right of the main subject).
3. Rate your confidence in each answer (high/medium/low)."""

response = client.chat.completions.create(
    model=model_name,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": sample_data_url}},
            {"type": "text", "text": spatial_prompt},
        ]
    }],
    max_tokens=400,
)
print(response.choices[0].message.content)

### Question

Consider this evaluation result from a radiology VLM:

In [ ]:
results = {
    "cardiomegaly_sensitivity": 0.95,   # correctly identifies enlarged heart
    "pleural_effusion_FPR": 0.08,       # falsely reports effusion in 8% of normal X-rays
    "n_chest_xrays": 50000,
    "n_pathology_slides": 200,
}

1. The model has a false positive rate of 8% for pleural effusion on normal X-rays. What type of failure mode is this, and why is it particularly dangerous in a clinical setting?
2. Should you fine-tune a general-purpose VLM (like LLaVA) on your radiology data, or use a domain-specific model (like BiomedCLIP) as the vision encoder? What factors would influence this decision?
3. You have 50,000 chest X-rays but only 200 pathology slides. How would you design a model that works for both imaging types?

### Answer

1. This is **hallucination of image content**, a false positive where the model describes a finding that is not present. In a clinical setting, this is dangerous because it could lead to unnecessary follow-up tests, treatments, or patient anxiety. Unlike text hallucination (where the model makes up facts), visual hallucination is harder for a non-expert to catch because it requires reviewing the original image.
2. Key factors: **Data availability** (if you have enough radiology data, fine-tuning a general VLM can work well; with limited data, a domain-specific encoder may generalize better). **Task specificity** (if you need to detect subtle radiological findings, a vision encoder pretrained on medical images will have better features than one trained on natural images). **Compute budget** (fine-tuning a large VLM is expensive; using a pretrained domain-specific encoder is cheaper). In practice, a hybrid approach often works best: use BiomedCLIP as the vision encoder and fine-tune only the projection layer and LLM.
3. **Transfer learning across imaging domains**: Pretrain the vision encoder on the 50,000 chest X-rays (where you have abundant data). Then fine-tune on the 200 pathology slides, leveraging the visual features learned from X-rays. While X-rays and pathology slides look different, lower-level features (edges, textures, contrast patterns) transfer across imaging types. Use data augmentation and LoRA-based fine-tuning to prevent overfitting on the small pathology dataset.

## Summary

| Concept | Key Idea |
|---------|----------|
| Vision Transformer (ViT) | Split image into patches, embed and process with a standard Transformer |
| Patch embedding | Image patches are analogous to text tokens: flatten and project |
| CLIP | Contrastive learning aligns image and text in a shared embedding space |
| Zero-shot classification | Classify images using text prompts without task-specific training |
| VLM architecture | Vision encoder + projection layer + LLM decoder |
| LLaVA | Open-source VLM: CLIP ViT + MLP projector + Vicuna LLM |
| Multi-modal fusion | Combining features from multiple modalities improves predictions |
| BiomedCLIP | CLIP trained on 15M biomedical image-text pairs from PubMed |
| Token cost | Images add hundreds of tokens; $O(n^2)$ attention makes this expensive |
| Modality universality | The Transformer processes any modality once it is tokenized into a sequence |

## Recommended Resources

* Dosovitskiy et al. (2020), ["An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale"](https://arxiv.org/abs/2010.11929) -- the ViT paper
* Radford et al. (2021), ["Learning Transferable Visual Models From Natural Language Supervision"](https://arxiv.org/abs/2103.00020) -- the CLIP paper
* Liu et al. (2023), ["Visual Instruction Tuning"](https://arxiv.org/abs/2304.08485) -- the LLaVA paper
* Jay Alammar, ["The Illustrated Stable Diffusion"](https://jalammar.github.io/illustrated-stable-diffusion/) -- visual guide to diffusion models (next lecture topic)
* Zhang et al. (2023), ["BiomedCLIP: A Multimodal Biomedical Foundation Model"](https://arxiv.org/abs/2303.00915) -- CLIP for biomedical data
* Lilian Weng, ["Generalized Visual Language Models"](https://lilianweng.github.io/posts/2022-06-09-vlm/) -- comprehensive blog post on VLMs
* [Hugging Face Vision Transformers guide](https://huggingface.co/docs/transformers/model_doc/vit) -- practical guide to using ViT